In [ ]:
"""data_explore.py
This notebook is meant only for preliminary data exploration and will not 
be used in model training. 
"""

# Standard imports 
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_theme("paper", "whitegrid")

In [ ]:
# Read data
project_path = "../.."
data_dir = os.path.join(project_path, "data"); 
thermal_data_path = os.path.join(data_dir, "thermal", "thermal_all_data.csv")
solvent_data_path = os.path.join(data_dir, "solvent", "solvent_all_data.csv")

thermal_all_df = pd.read_csv(thermal_data_path)
solvent_all_df = pd.read_csv(solvent_data_path)
thermal_all_df.head()

## Thermal Stability Data

### f- Data 

- f- data are f-scores? 
- f-chi: medians 177, 330, 793, 916, left-skewed
- f-Z: medians 3812, 7351, 10530, 9632, left-skewed
- f-I: medians 20 48 90 116, left-skewed
- f-T: medians 144, 407, 579, 556, left-skewed
- f-S: ...

### Other Data 

These data are likely related to some distribution, or some chemical attributes extracted from the cif files. 

- mc-: McNemar's test? 
- D_mc-: ??
- f-lig-: ??
- lc-: ??
- D_lc-: ??
- func-: ??
- D_func-: ??
- Df, Di, Dif: some kind of distribution data ??

### MOF Features 

These must have been extracted from the cif files. See [feature_extraction.py](../model_features/feature_extraction.py). 

GPOAV, GPONAV, GPOV, GSA, POAV, POAV_vol_frac, PONAV, PONAV_vol_frac, VPOV, VSA, cell_v

## Labels 

- T: thermal breakdown temperature

### Discard 

- filename 
- 0 
- CoRE_name
- refcode

In [ ]:
thermal_feature_names = thermal_all_df.columns

# Filter columns that start with "f-chi"
prefix_columns = [col for col in thermal_feature_names if col.startswith("f-T")]
num_columns = len(prefix_columns)
n = int(np.ceil(np.sqrt(num_columns)))

# Create subplots
fig1, axes1 = plt.subplots(n, n, figsize=(15, 15))
axes1 = axes1.flatten()

# Plot histograms
for i, col in enumerate(prefix_columns):
    sns.histplot(thermal_all_df[col], ax=axes1[i], kde=True)
    axes1[i].set_title(col + f" | median: {thermal_all_df[col].median()} | max: {thermal_all_df[col].max()}")

for j in range(i + 1, n * n):
    fig1.delaxes(axes1[j])
plt.tight_layout()

In [ ]:
# Correlation 
sns.heatmap(thermal_all_df.corr())

In [ ]:
# Export raw features: features that I am certain are from the cif files 
thermal_raw_df = thermal_all_df[["GPOAV", "GPONAV", "GPOV", "GSA", "POAV", "POAV_vol_frac", "PONAV", "PONAV_vol_frac", "VPOV", "VSA", "cell_v"]]
# thermal_raw_df.to_pickle(os.path.join(data_dir, "thermal_raw.pickle"))

### Solvent removal stability model

In [ ]:
solvent_all_df.head()

In [ ]:
removed_cols = ["Unnamed: 0", "doi", "filename", "0", "CoRE_name", "refcode", "name"]
solvent_clean_df = solvent_all_df.loc[:, ~solvent_all_df.columns.isin(removed_cols)]
solvent_clean_df.columns[solvent_clean_df.isna().any()].tolist() # List columns with nan values
solvent_clean_df.head()

In [ ]:
solvent_test_data_path = os.path.join(data_dir, "solvent", "solvent_test_data.pkl")
solvent_test_data = pd.read_pickle(solvent_test_data_path)

### Water & Hazardous Environment Model

In [16]:
# Read data
project_path = "../.."
data_dir = os.path.join(project_path, "data"); 
water_and_haz_data_path = os.path.join(data_dir, "water_and_haz", "data.csv"); 
water_and_haz_df = pd.read_csv(water_and_haz_data_path)
water_and_haz_df.head()

,MOF_name,data_set,water_label,acid_label,base_label,boiling_label,Di,Df,Dif,VSA,...,f-lig-T-2,f-lig-T-3,f-lig-Z-0,f-lig-Z-1,f-lig-Z-2,f-lig-Z-3,f-lig-chi-0,f-lig-chi-1,f-lig-chi-2,f-lig-chi-3
0,ACOFUU,CoRE,2,0,0,0,7.99458,6.67411,7.99458,2280.100,...,1696.000000,2024.000000,2032.000000,4616.0,6536.0,7248.000000,540.553600,1176.9320,1966.5344,2565.196800
1,ACOGAB,CoRE,2,0,0,0,5.20086,3.89327,4.83563,1877.510,...,1696.000000,2024.000000,2032.000000,4616.0,6536.0,7248.000000,540.553600,1176.9320,1966.5344,2565.196800
2,ACOGEF,CoRE,2,0,0,0,5.20116,3.94888,4.87431,1842.000,...,1696.000000,2024.000000,2032.000000,4616.0,6536.0,7248.000000,540.553600,1176.9320,1966.5344,2565.196800
3,ADABUE,CoRE,4,1,1,1,4.54949,3.31506,4.54949,772.133,...,240.666667,221.333333,457.333333,940.0,1222.0,957.333333,106.017633,214.7524,320.4522,323.706533
4,AFEJUQ,CoRE,2,0,0,0,5.87037,4.28862,5.87037,2044.710,...,352.000000,360.000000,466.000000,1080.0,1400.0,1288.000000,122.228200,268.8210,406.2200,458.812000


In [20]:
removed_cols = ["MOF_name", "data_set", "Unnamed: 0", "doi", "filename", "0", "CoRE_name", "refcode", "name"]
water_and_haz_df = water_and_haz_df.loc[:, ~water_and_haz_df.columns.isin(removed_cols)]
water_and_haz_df.describe()

,water_label,acid_label,base_label,boiling_label,Di,Df,Dif,VSA,GSA,VPOV,...,f-lig-T-2,f-lig-T-3,f-lig-Z-0,f-lig-Z-1,f-lig-Z-2,f-lig-Z-3,f-lig-chi-0,f-lig-chi-1,f-lig-chi-2,f-lig-chi-3
count,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,...,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000,1092.000000
mean,2.606227,0.080586,0.056777,0.060440,6.982168,4.429995,6.849312,1148.604708,1159.054228,0.314410,...,1328.567070,1354.965130,880.402258,1913.329894,2831.886307,2963.807279,198.234919,432.930249,735.515906,911.451338
std,0.762718,0.272323,0.231521,0.238409,4.852269,3.316705,4.821749,918.768869,1152.208076,0.227097,...,20913.014121,19941.332118,683.737687,3032.184849,5876.054690,6608.700480,190.526076,826.596602,2177.797682,3416.792974
min,1.000000,0.000000,0.000000,0.000000,1.561290,0.662930,1.407400,0.000000,0.000000,0.000000,...,0.000000,0.000000,85.000000,84.000000,0.000000,0.000000,15.744100,15.504000,0.000000,0.000000
25%,2.000000,0.000000,0.000000,0.000000,4.103580,2.518797,4.012258,0.000000,0.000000,0.105100,...,256.000000,250.000000,549.090909,1036.000000,1480.000000,1426.000000,118.714400,227.868000,355.476400,385.387000
50%,3.000000,0.000000,0.000000,0.000000,5.653210,3.763595,5.560025,1342.540000,1040.585000,0.314900,...,430.919540,458.000000,711.000000,1440.000000,2080.000000,2080.089552,160.889050,326.695767,527.469600,598.458633
75%,3.000000,0.000000,0.000000,0.000000,8.188040,5.585170,8.024560,1933.452500,1840.832500,0.484750,...,697.000000,746.875000,1029.000000,2161.000000,3139.750000,3366.800000,229.175905,478.724250,787.241200,945.799333
max,4.000000,1.000000,1.000000,1.000000,39.110620,37.476400,39.110620,3539.340000,5109.460000,0.889300,...,686300.666667,653072.000000,12144.000000,81457.333333,165812.666667,176592.000000,3640.455467,23104.400333,65858.167467,104238.124000
